# «Тишина АЗ-26» — решение (v2: + маркеры границ эпизодов)

**Идея.** Предотказные эпизоды (~20 точек, ~4.7% строк) почти не видны по уровню отдельных датчиков:
значения остаются в штатных границах. Но внутри эпизода у многих датчиков одновременно
растёт *локальный шум* (дисперсия первой разности), дисперсия уровня в окне относительно окружения,
а в конце эпизода часто бывает «возврат» скачком. Сигнал размазан по всем 15 датчикам, поэтому:

1. для каждого датчика в сеансе считаем оконные статистики (центрированные окна — разметка ретроспективная,
   поэтому можно смотреть и в прошлое, и в будущее);
2. агрегируем их **по датчикам** (mean / max / q80 / top-3 / счётчики) — это устойчивее, чем 400+ посенсорных признаков;
3. добавляем ранги агрегатов внутри сеанса (нормировка между сеансами) и признаки положения относительно смены режима;
4. LightGBM с сильной регуляризацией, валидация GroupKFold по `session_id`;
5. сглаживаем предсказание скользящим средним по времени внутри сеанса (эпизоды — непрерывные отрезки).

In [1]:
import numpy as np, pandas as pd, lightgbm as lgb, warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
warnings.filterwarnings('ignore')
print('lightgbm', lgb.__version__, '| pandas', pd.__version__)

train = pd.read_csv('telemetry_train.csv')
test = pd.read_csv('telemetry_test.csv')
print(train.shape, test.shape, 'доля аномалий:', round(train.is_anomaly.mean(), 4))
print('сеансов train/test:', train.session_id.nunique(), test.session_id.nunique())

lightgbm 4.7.0 | pandas 3.0.6
(120000, 20) (40000, 19) доля аномалий: 0.0475
сеансов train/test: 32 16


## Признаки

In [2]:
import numpy as np, pandas as pd

SENS = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator',
        'p_fuel', 'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z', 'signal_strength']
HW = (4, 8, 12, 20)   # полуширины «внутреннего» окна
CTX = 30              # длина контекста слева и справа


def _wsum(v, lo, hi):
    """Сумма v[i+lo : i+hi] для каждого i (NaN -> 0), окна обрезаются по краям."""
    n = len(v)
    cs = np.r_[0, np.cumsum(v)]
    i = np.arange(n)
    a = np.clip(i + lo, 0, n); b = np.clip(i + hi, 0, n)
    return cs[b] - cs[a]


def _wmean(x, lo, hi):
    m = ~np.isnan(x)
    s = _wsum(np.where(m, x, 0), lo, hi); c = _wsum(m.astype(float), lo, hi)
    s2 = _wsum(np.where(m, x * x, 0), lo, hi)
    mu = s / np.maximum(c, 1)
    var = np.maximum(s2 / np.maximum(c, 1) - mu ** 2, 0)
    mu[c == 0] = np.nan
    return mu, var, c


def session_feats(d):
    d = d.reset_index(drop=True)
    n = len(d); pos = np.arange(n)
    F = {}
    F['mode'] = d['mode'].map({'cruise': 0, 'maneuver': 1, 'eclipse': 2}).values
    ch = np.r_[0, np.where(d['mode'].values[1:] != d['mode'].values[:-1])[0] + 1, n]
    k = np.searchsorted(ch, pos, side='right')
    F['dist_prev_mode'] = pos - ch[k - 1]
    F['dist_next_mode'] = ch[np.minimum(k, len(ch) - 1)] - pos
    F['rel_pos'] = pos / n

    agg = {}
    for s in SENS:
        raw = d[s].values.astype(float)
        dx = np.diff(raw, prepend=np.nan)
        mad = np.nanmedian(np.abs(dx - np.nanmedian(dx)))
        ns = 1.4826 * mad + 1e-12                 # шум по первой разности
        z = raw / ns
        adz = np.abs(dx / ns)
        # скачки: максимум |dz| в окнах вперёд/назад
        a0 = np.nan_to_num(adz)
        sa = pd.Series(a0)
        for w in (10, 25):
            f = sa[::-1].rolling(w, min_periods=1).max()[::-1].shift(-1).values
            b = sa.rolling(w, min_periods=1).max().values
            F[f'{s}_jmpf{w}'] = f; F[f'{s}_jmpb{w}'] = b
        F[f'{s}_vol11'] = pd.Series(adz).rolling(11, center=True, min_periods=1).mean().values
        # «горб»: среднее внутри окна против контекста слева+справа
        for h in HW:
            mi, vi, ci = _wmean(z, -h, h + 1)
            ml, vl, cl = _wmean(z, -h - CTX, -h)
            mr, vr, cr = _wmean(z, h + 1, h + 1 + CTX)
            mc = np.where(np.isnan(ml), mr, np.where(np.isnan(mr), ml, (ml * cl + mr * cr) / np.maximum(cl + cr, 1)))
            vc = (vl * cl + vr * cr) / np.maximum(cl + cr, 1) + 1e-9
            bump = (mi - mc) / np.sqrt(vc) * np.sqrt(np.maximum(ci, 1))
            ratio = np.sqrt(vi / vc)
            lr = (mr - ml) / np.sqrt(vc)
            F[f'{s}_bump{h}'] = bump
            F[f'{s}_vr{h}'] = ratio
            F[f'{s}_lr{h}'] = lr
            agg.setdefault(f'bump{h}', []).append(np.abs(np.nan_to_num(bump)))
            agg.setdefault(f'vr{h}', []).append(np.nan_to_num(ratio, nan=1))
        # уровень шума: дисперсия первой разности в окне против длинного контекста
        e2 = (dx / ns) ** 2
        for h in (6, 10, 16):
            m, _, _ = _wmean(e2, -h, h + 1)
            for C in (60, 150):
                mc, _, _ = _wmean(e2, -h - C, h + C + 1)
                v = np.log(m + 1e-3) - np.log(mc + 1e-3)
                F[f'{s}_dvr{h}_{C}'] = v
                agg.setdefault(f'dvr{h}_{C}', []).append(np.nan_to_num(v))
        # дисперсия уровня в окне против контекста 80
        for h in (6, 10, 14):
            mi, vi, ci = _wmean(z, -h, h + 1)
            ml, vl, cl = _wmean(z, -h - 80, -h)
            mr, vr_, cr = _wmean(z, h + 1, h + 81)
            vc = (vl * cl + vr_ * cr) / np.maximum(cl + cr, 1) + 1e-9
            v = np.log(vi + 1e-9) - np.log(vc)
            F[f'{s}_lvr{h}'] = v
            agg.setdefault(f'lvr{h}', []).append(np.nan_to_num(v))
        med = pd.Series(z).rolling(151, center=True, min_periods=1).median().values
        sm = pd.Series(z).rolling(9, center=True, min_periods=1).mean().values
        F[f'{s}_dev151'] = sm - med
        agg.setdefault('dev151', []).append(np.abs(np.nan_to_num(F[f'{s}_dev151'])))
        for w in (10, 25):
            agg.setdefault(f'jmpf{w}', []).append(F[f'{s}_jmpf{w}'])
            agg.setdefault(f'jmpb{w}', []).append(F[f'{s}_jmpb{w}'])
        agg.setdefault('lr8', []).append(np.abs(np.nan_to_num(F[f'{s}_lr8'])))
        agg.setdefault('vol11', []).append(np.nan_to_num(F[f'{s}_vol11']))
    for k_, v in agg.items():
        A = np.vstack(v)
        F[f'agg_{k_}_mean'] = A.mean(0)
        F[f'agg_{k_}_max'] = A.max(0)
        F[f'agg_{k_}_q80'] = np.quantile(A, 0.8, axis=0)
        F[f'agg_{k_}_top3'] = np.sort(A, 0)[-3:].mean(0)
        if k_.startswith(('dvr', 'lvr')):
            F[f'agg_{k_}_n05'] = (A > 0.5).sum(0)
            F[f'agg_{k_}_n10'] = (A > 1.0).sum(0)
    out = pd.DataFrame(F)
    out['timestamp_id'] = d['timestamp_id'].values
    out['session_id'] = d['session_id'].values
    return out


def build(df):
    return pd.concat([session_feats(g) for _, g in df.groupby('session_id', sort=False)], ignore_index=True)


### Маркеры границ эпизодов (v2)
Разбор ошибок показал, что границы эпизодов резкие: в первой строке эпизода и в первой штатной строке после него
часто скачут **сразу несколько датчиков** (≥2 датчика с |Δ|>4σ: 2.8% случайных строк против 16% начал и 25% концов).
Добавляем счётчики одновременных скачков, их скользящие средние и расстояния до ближайших таких строк.

In [3]:

SENS_ = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator',
        'p_fuel', 'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z', 'signal_strength']
EDGES = np.array([0, .25, .5, .75, 1, 1.5, 2, 2.5, 3, 4, 5, 6, 8, 11, 15, 25, np.inf])


def norm_diffs(df, per_session=True):
    """|первая разность| / робастный масштаб шума; NaN там, где разность не определена."""
    g = df.session_id.values
    X = df[SENS_].values.astype(float)
    D = np.diff(X, axis=0, prepend=np.nan)
    D[np.r_[True, g[1:] != g[:-1]]] = np.nan
    Z = np.empty_like(D)
    for s in np.unique(g):
        m = g == s
        d = D[m]
        ns = 1.4826 * np.nanmedian(np.abs(d - np.nanmedian(d, 0)), 0) + 1e-12
        Z[m] = np.abs(d / ns)
    return Z




def bracket_feats(df):
    """Признаки одновременных скачков нескольких датчиков (маркеры границ эпизода)."""
    Z = np.nan_to_num(norm_diffs(df))
    g = df.session_id.values
    F = {}
    for thr in (3, 4, 6):
        F[f'cnt{thr}'] = (Z > thr).sum(1).astype(float)
    F['sumlog'] = np.log1p(Z).sum(1)
    out = pd.DataFrame(F)
    res = {}
    for s in pd.unique(g):
        m = np.where(g == s)[0]; n = len(m); pos = np.arange(n)
        R = {}
        for thr, k in ((3, 2), (4, 2), (4, 1), (6, 1)):
            ev = out[f'cnt{thr}'].values[m] >= k
            idx = np.where(ev)[0]
            j = np.searchsorted(idx, pos, side='right')
            prev = np.where(j > 0, idx[np.maximum(j - 1, 0)], -10 ** 4)       # событие на строке <= t (начало)
            nxt = np.where(j < len(idx), idx[np.minimum(j, len(idx) - 1)], 10 ** 4 + n)  # событие > t (конец)
            dp = np.minimum(pos - prev, 200); dn = np.minimum(nxt - pos, 200); span = np.minimum(nxt - prev, 400)
            key = f'{thr}_{k}'
            R[f'dprev_{key}'] = dp; R[f'dnext_{key}'] = dn; R[f'span_{key}'] = span
            R[f'brk_{key}'] = ((span >= 6) & (span <= 60)).astype(float)
        for c in ('cnt3', 'cnt4', 'sumlog'):
            v = pd.Series(out[c].values[m])
            for w in (15, 31):
                R[f'{c}_m{w}'] = v.rolling(w, center=True, min_periods=1).mean().values
        res[s] = pd.DataFrame(R, index=m)
    return pd.concat([out, pd.concat(res.values()).sort_index()], axis=1)


In [4]:
Xtr = build(train)
Xte = build(test)
assert (Xtr.timestamp_id.values == train.timestamp_id.values).all()
assert (Xte.timestamp_id.values == test.timestamp_id.values).all()

BASE = ['mode', 'dist_prev_mode', 'dist_next_mode', 'rel_pos']
AGG = [c for c in Xtr.columns if c.startswith('agg')]

def make_matrix(X):
    R = X[AGG].groupby(X.session_id.values).rank(pct=True).add_suffix('_rk')   # ранг внутри сеанса
    return pd.concat([X[AGG + BASE], R], axis=1)

Btr = bracket_feats(train).reset_index(drop=True)
Bte = bracket_feats(test).reset_index(drop=True)

XX_tr = pd.concat([make_matrix(Xtr), Btr], axis=1)
XX_te = pd.concat([make_matrix(Xte), Bte], axis=1)
y, groups = train.is_anomaly.values, train.session_id.values
print('признаков:', XX_tr.shape[1])

признаков: 258


## Валидация (GroupKFold по сеансам)

In [5]:
PARAMS = dict(objective='binary', learning_rate=0.02, num_leaves=15, min_child_samples=400,
              feature_fraction=0.2, bagging_fraction=0.7, bagging_freq=1, lambda_l2=30,
              verbose=-1, n_jobs=4)
ROUNDS, SMOOTH, SEEDS = 400, 15, [0, 1, 2, 3, 4]

def smooth(p, sess, w=SMOOTH):
    return pd.Series(p).groupby(np.asarray(sess)).transform(
        lambda v: v.rolling(w, center=True, min_periods=1).mean()).values

oof = np.zeros(len(y))
for k, (a, b) in enumerate(GroupKFold(8).split(XX_tr, y, groups)):
    m = lgb.train(dict(PARAMS, seed=0), lgb.Dataset(XX_tr.iloc[a], y[a]), ROUNDS)
    oof[b] = m.predict(XX_tr.iloc[b])
    print(f'fold {k}: AUC={roc_auc_score(y[b], oof[b]):.4f}  (сглаж. {roc_auc_score(y[b], smooth(oof[b], groups[b])):.4f})')
print(f'OOF ROC-AUC: {roc_auc_score(y, oof):.4f}')
print(f'OOF ROC-AUC после сглаживания (окно {SMOOTH}): {roc_auc_score(y, smooth(oof, groups)):.4f}')

fold 0: AUC=0.7455  (сглаж. 0.7506)


fold 1: AUC=0.6818  (сглаж. 0.6901)


fold 2: AUC=0.7761  (сглаж. 0.7807)


fold 3: AUC=0.6426  (сглаж. 0.6349)


fold 4: AUC=0.7088  (сглаж. 0.7050)


fold 5: AUC=0.7568  (сглаж. 0.7696)


fold 6: AUC=0.6863  (сглаж. 0.7013)


fold 7: AUC=0.7444  (сглаж. 0.7453)
OOF ROC-AUC: 0.7153
OOF ROC-AUC после сглаживания (окно 15): 0.7194


## Обучение на всей выборке и сабмит

In [6]:
pred = np.zeros(len(XX_te))
for s in SEEDS:
    m = lgb.train(dict(PARAMS, seed=s), lgb.Dataset(XX_tr, y), ROUNDS)
    pred += m.predict(XX_te) / len(SEEDS)
pred = smooth(pred, test.session_id.values)

sub = pd.DataFrame({'timestamp_id': test.timestamp_id, 'anomaly_prob': pred})
sub.to_csv('submission_v2.csv', index=False)
print(sub.shape, sub.anomaly_prob.describe().round(4).to_dict())
sub.head()

(40000, 2) {'count': 40000.0, 'mean': 0.0443, 'std': 0.0577, 'min': 0.0064, '25%': 0.022, '50%': 0.032, '75%': 0.048, 'max': 0.9535}


,timestamp_id,anomaly_prob
0,220001,0.031174
1,220002,0.031976
2,220003,0.032239
3,220004,0.032431
4,220005,0.033662


In [7]:
imp = pd.Series(m.feature_importance('gain'), XX_tr.columns).sort_values(ascending=False)
imp.head(20).round(1)

agg_dvr6_150_max        13793.0
span_3_2                11818.1
dist_next_mode          10936.3
agg_jmpf10_max_rk       10711.0
agg_dvr10_60_n10_rk      9973.7
agg_jmpf25_top3_rk       8711.0
rel_pos                  8423.7
agg_jmpf25_max_rk        8007.3
span_4_2                 7540.8
agg_jmpf25_mean_rk       7432.8
span_6_1                 6735.6
agg_lr8_q80_rk           6230.9
agg_dvr6_150_top3        6085.3
agg_dvr6_150_top3_rk     5895.0
agg_dvr10_150_max        5609.4
agg_jmpb25_max_rk        5408.3
agg_lr8_mean_rk          5254.6
span_4_1                 5150.9
dist_prev_mode           4914.4
agg_dvr10_60_max_rk      4397.3
dtype: float64